# 00 — Quadruped-PyMPCの背景・目的・結論と学習地図

## 1. このリポジトリはなぜ作られたか

四足歩行では、目標速度を与えるだけでは脚は動きません。「いつ接地するか」「次にどこへ着地するか」「各接地点で何Nの床反力を出すか」「その力をどの関節トルクで作るか」を、転倒しない速さで繰り返し決める必要があります。

`iit-DLSLab/Quadruped-PyMPC` は、この問題を **Single Rigid Body Dynamics（SRBD）に基づくModel Predictive Control** として解く研究・実装リポジトリです。標準経路はCasADiで非線形モデルを記述し、acadosで有限時間最適制御問題を反復的に解きます。MuJoCo上の複数四足ロボットに加え、公開READMEではUnitree系実機との接続も想定されています。

## 2. 目的

この教材の目的はAPIの使い方だけを覚えることではありません。現行コードを正本として、

1. 速度指令から接地列・着地点参照が作られる理由
2. SRBDの式がCasADi変数へどう対応するか
3. Q/R、摩擦円錐、接触制約がacados OCPへどう入るか
4. 最適床反力がJacobian転置で関節トルクへ変換される理由
5. 症状からチューニング箇所を選び、式をテスト付きで変更する方法

を、処理ブロックごとに理解することです。

## 3. 先に結論

このリポジトリの強みは、**Gait → foothold → centroidal NMPC → stance/swing torque → MuJoCo** がPythonで一気通貫し、gradient-based MPCとsampling-based MPCの研究分岐も同じ枠組みに置かれていることです。一方、標準nominalモデルは脚の全身力学を予測するモデルではなくSRBD近似です。性能は地形、速度、摩擦モデル、gait、トルク飽和、solver周期の組合せに依存し、READMEの機能一覧だけでは判断できません。そのため最後に実コードを30シナリオで計測します。

**前提**: なし

> 各章は「背景 → ASCIIデータ流 → 数式 → コメント付きコード → 実行結果 → 限界」の順に読みます。`実装事実` と`学習用近似`を混同しません。

In [1]:
# 背景: 各章を同じ作業ディレクトリと依存関係で再実行するには、リポジトリの基準パスと外部実装の場所を最初に固定する必要がある。
# 目的: Quadruped-PyMPCをimport可能にし、acadosとヘッドレスMuJoCoの実行環境を後続セルへ引き渡す。
# ファイルシステム上の基準パスを型安全に扱うためPathを読み込む。
from pathlib import Path
# 環境変数の設定とPythonのモジュール探索パス更新に必要な標準ライブラリを読み込む。
import os, sys

# Notebookを起動した現在位置を絶対パスへ正規化し、教材全体の基準候補とする。
ROOT = Path.cwd().resolve()
# notebook_pympc直下から起動した場合だけ、リポジトリ直下へ基準を1階層戻す。
if ROOT.name == "notebook_pympc":
    # externalディレクトリを参照できるリポジトリ直下へROOTを合わせる。
    ROOT = ROOT.parent
# 上流制御実装が置かれたQuadruped-PyMPCの絶対パスを構成する。
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
# 誤った起動位置のまま進まず、依存リポジトリの欠落を具体的なパス付きで検出する。
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
# 同じパスを重複登録せず、まだimport探索対象でない場合だけ追加する。
if str(PYMPC_ROOT) not in sys.path:
    # ローカルのquadruped_pympcパッケージを通常のimport文で読めるよう探索順の先頭へ置く。
    sys.path.insert(0, str(PYMPC_ROOT))

# acados生成物・共有資源の基準位置を未設定時だけ登録し、利用者の明示設定は上書きしない。
os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
# 画面のない環境でもMuJoCoを描画可能にするため、未設定時のOpenGL backendをEGLにする。
os.environ.setdefault("MUJOCO_GL", "egl")
# 実際に採用されたworkspace基準を表示し、相対パス問題を診断できるようにする。
print("workspace :", ROOT)
# import対象となる上流実装の場所を表示し、参照しているコード版を確認可能にする。
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 4. 1制御周期のASCIIデータフロー

```text
┌─────────────────────────────────────────────────────────────────────────────┐
│ User / command generator                                                   │
│ 目標: v_ref(W) [3] m/s, omega_ref [3] rad/s                               │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ MuJoCo / gym-quadruped plant                                                │
│ 観測: COM, base姿勢・速度, 足位置・速度, q, qdot, J, M, contact             │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ WBInterface — reference generation                                         │
│  ├─ PeriodicGaitGenerator: phase φ_i → contact c_i,k ∈ {0,1} [4×N]         │
│  ├─ FootholdReferenceGenerator: p_hip, v, v_ref → p_foot_ref                │
│  └─ SwingTrajectoryController: lift-off → spline → p, pdot, pddot           │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ SRBDControllerInterface                                                    │
│ state/ref/contact/inertia を nominal・input_rates・sampling等へ振り分ける   │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ Nominal centroidal NMPC (CasADi + acados)                                  │
│ min Σ||x-x_ref||²_Q + ||u-u_ref||²_R                                       │
│ s.t. SRBD, contact mask, friction cone, GRF/foothold constraints            │
│ 出力: 第0予測段の GRF F_i [12] N と optimized foothold [4×3] m             │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ Stance / swing torque                                                       │
│ stance: tau_i = -J_i^T F_i   swing: Cartesian PD + feedback linearization  │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ actuator orderへ格納 → 90% soft torque clip → env.step(action)             │
└──────────────────────────────┬──────────────────────────────────────────────┘
                               └──────── 次の観測へ戻る ───────────────────────┘
```

式で縮めると、

\[
v^{ref}\rightarrow (c_{i,k},p^{ref}_{foot})
\rightarrow \operatorname{NMPC}(x,u)
\rightarrow F_i^{cmd}\rightarrow \tau_i\rightarrow x_{next}
\]

です。標準`nominal` MPCが直接最適化する入力は、足速度12要素と床反力12要素です。関節トルクは後段で計算されます。

In [2]:
# 背景: 四足歩行MPCは参照生成・力学・最適化・トルク変換が連鎖するため、依存順を無視すると式と実装の対応を見失いやすい。
# 目的: 章01〜16を上流信号から検証・改造へ進む推奨順で一覧表示し、学習経路を確認する。
# 各章名を依存関係の順に並べ、表示と後続の進捗管理に再利用できるリストへまとめる。
learning_order = [
    "01 environment/source map",  # 実行環境と正本コードを最初に固定する章。
    "02 vectors, units, frames",  # shape・単位・座標系という全章共通の契約を学ぶ章。
    "03 MuJoCo plant",  # 制御入力を受ける全身物理モデルを確認する章。
    "04 closed-loop timing",  # simulatorとMPCの異なる時間刻みを整理する章。
    "05 gait/contact schedule",  # 位相から4脚×予測段の接触列を作る章。
    "06 foothold reference",  # 速度指令からworld座標の着地点を作る章。
    "07 swing trajectory",  # 離地から着地までの足先位置・速度・加速度を作る章。
    "08 SRBD dynamics",  # 接触力と胴体加速度を結ぶ縮約力学を学ぶ章。
    "09 friction/contact constraints",  # 実現可能な床反力を定める制約を学ぶ章。
    "10 MPC objective",  # 状態誤差と入力を評価するQ/Rコストを学ぶ章。
    "11 receding horizon/acados",  # 有限区間問題を反復して解く実装を学ぶ章。
    "12 force-to-torque conversion",  # 最適床反力を関節トルク[N·m]へ写す章。
    "13 end-to-end baseline",  # 全ブロックを結合した基準動作を再現する章。
    "14 logging and diagnosis",  # 症状を信号へ分解して原因を診断する章。
    "15 tuning laboratory",  # パラメータ変更を比較実験で評価する章。
    "16 equation modification capstone",  # 式・実装・テストを一体で改造する章。
]
# enumerateを1始まりで使い、教材の章番号とリスト要素を対応させて走査する。
for i, item in enumerate(learning_order, 1):
    # 章番号を2桁に揃えて表示し、順序を視覚的に追いやすくする。
    print(f"{i:02d}. {item}")

01. 01 environment/source map
02. 02 vectors, units, frames
03. 03 MuJoCo plant
04. 04 closed-loop timing
05. 05 gait/contact schedule
06. 06 foothold reference
07. 07 swing trajectory
08. 08 SRBD dynamics
09. 09 friction/contact constraints
10. 10 MPC objective
11. 11 receding horizon/acados
12. 12 force-to-torque conversion
13. 13 end-to-end baseline
14. 14 logging and diagnosis
15. 15 tuning laboratory
16. 16 equation modification capstone


## 進級条件

- **理解**: shape・単位・frameを添えて信号を説明できる
- **再現**: Notebookを上から再実行して同じ結論になる
- **調整**: 一度に1群だけ変更し、仮説と評価量を先に書く
- **改造**: 式、CasADi式、制約、テストを同じ変更単位で扱う

`14` までは上流コードを変更しません。`15` は設定値の比較、`16` はNotebook内で
式の候補を検証します。上流ファイルの変更は、比較テストができてからです。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。